In [ ]:
#| default_exp reports_interactive

In [ ]:
#| export
import panel as pn
import pandas as pd

In [ ]:
#| export
from collections import defaultdict
import datetime as dt

In [ ]:
#| export
import holoviews as hv

In [ ]:
#| export
from portfolio.plots import timeseries_plot

In [ ]:
#| export
from bokeh.models.formatters import NumeralTickFormatter

In [ ]:
#| export
from portfolio.portfolio import *

In [ ]:
#| export
pn.extension('tabulator')

In [ ]:
from portfolio.sample_data import *

To create a report you simply pass it one or more portfolios

In [ ]:
rets, rf, cpi = sample_data_se()

In [ ]:
p = Portfolio('60/40', rets, {'bonds': .40, 'stocks': .60}, rf=rf, cpi=cpi)
p

## Return Driver Widget

In [ ]:
#| export
def return_drivers_w(p):
    r = p.return_drivers()
    avg = r.mean()
    return pn.Column(timeseries_plot(r - avg, hline=0, title='Rolling Excess Return Mean Centered'),
    pd.DataFrame(avg, columns=['value']).hvplot.bar(title='Avg Excess Return over Period', yformatter=NumeralTickFormatter(format="0.0%"), hover_tooltips=[("Series", "@index"), ("Value", "@value{0.00%}")]))

In [ ]:
#return_drivers_w(p)

## Time Selection Widget

In [ ]:
#| export
def time_period_w():
    drp = pn.widgets.DateRangePicker(name='Date Range', value=(dt.date(1982,1,1), dt.date(2025,1,1)))
    btn = pn.widgets.Button(name='Apply', button_type='primary')
    return drp, btn

## Portfolio information Widget

In [ ]:
#| export
def _get_info(p):
    sources = {'cpi': p.full_cpi, 'rets': p.full_rets, 'rf': p.full_rf}
    df = pd.DataFrame({k: {'datastream start': df.index.min(), 'datastream end': df.index.max(), 'sz': len(df)} for k, df in sources.items()}).T
    return pn.widgets.Tabulator(df)

In [ ]:
#_get_info(p)

In [ ]:
#| export
def _allocation_w(p):
    w = pd.DataFrame(index=['Allocation'], data=[p.weights]) * 100
    return pn.widgets.Tabulator(
        w,
        formatters={n: {'type': 'money', 'symbol': '%', 'symbolAfter': True, 'precision': 1} for n in w.columns}
    )

In [ ]:
#_allocation_w(p)

In [ ]:
#| export
def portfolio_info_w(p):
    return pn.Tabs(
        ('Data Streams', _get_info(p)),
        ('Allocation', _allocation_w(p))
    )

In [ ]:
#portfolio_info_w(p)

## Decade by decade widget

In [ ]:
#| export
def decade_w(p):
    return pn.Column(
        pn.pane.Markdown('### Real 10y Return (ann)'), timeseries_plot(p.lost_decade(components=True), hline=0.0, ylabel='Return (ann)'),
        pn.pane.Markdown('### Decade by Decade real return'), timeseries_plot(p.r_by_decade(), logy=True, hline=1.0, xlabel='Years Held', ylabel='Return'),
    )

In [ ]:
#| export
def decade_comparison_w(*portfolios):
    p = portfolios[0]
    #decades = list(p.r_by_decade().columns)
    decades = sorted(set().union(*[p.r_by_decade().columns for p in portfolios]))
    decade_w = pn.widgets.Select(name='Decade', options=decades, value=decades[0])
    def plot(decade):
        df = pd.concat({p.name: p.r_by_decade()[decade] for p in portfolios if decade in p.r_by_decade().columns}, axis=1)
        return timeseries_plot(df, logy=True, is_perc=False, xlabel='Years Held', ylabel='Real Return (cumulative)', hline=1.0)
    return pn.Column(decade_w, pn.bind(plot, decade_w))

In [ ]:
#| export
def lost_decade_comparison_w(*portfolios):
    df = pd.concat({p.name: p.lost_decade() for p in portfolios}, axis=1)
    return timeseries_plot(df, hline=0, ylabel='Real 10y Return (ann)')

## Portfolio Overview Widget

In [ ]:
#| export
def portfolio_overview(p):
    asset_stats   = pn.bind(lambda p: pn.panel(p.asset_stats()), p)
    risk_contrib  = pn.bind(lambda p: pn.panel(p.risk_contribution()), p)
    roll_vol_plot = pn.bind(lambda p: timeseries_plot(p.rolling_vol()), p)
    rc_plot       = pn.bind(lambda p: timeseries_plot(p.rc_simple()), p)
    roll_no_exc   = pn.bind(lambda p: timeseries_plot(p.roll_return(excess=False, extras=True), hline=0), p)
    correlation   = pn.bind(lambda p: pn.panel(p.correlation()), p)
    info = pn.bind(lambda p: portfolio_info_w(p), p)

    decade_analysis = pn.bind(lambda p: decade_w(p), p)

    overview = pn.Column(
        pn.pane.Markdown('### Asset Stats'), asset_stats,
        pn.pane.Markdown('## Risk'),
        pn.pane.Markdown('### Risk Contribution'), risk_contrib,
        pn.pane.Markdown('### Rolling Volatility'), roll_vol_plot,
        pn.pane.Markdown('### Risk Contribution (Plot)'), rc_plot,
        pn.pane.Markdown('### Extra 12m Rolling (No Excess)'), roll_no_exc,
        pn.pane.Markdown('### Drawdowns Assets + Portfolio'), pn.bind(lambda p: timeseries_plot(p.drawdown_series(assets=True), hline=0), p),
    )
    return_drivers = pn.Column(pn.pane.Markdown('### Return Drivers 5 years (excess mean centered'), pn.bind(lambda p: return_drivers_w(p), p))
    corr = pn.Column(pn.pane.Markdown('### Correlation Matrix'), correlation)
    
    return pn.Tabs(
        ('Portfolio Overview', overview),
        ('Correlation Matrix', corr),
        ('Return Drivers', return_drivers),
        ('Info', info),
        ('Decade analysis', decade_analysis),
        )

## Strategy Comparison

We will follow a naming convention where a portfolio is named 'strategy_country'

In [ ]:
ps = [
    Portfolio('60/40_sweden', rets, {'bonds': .40, 'stocks': .60}, rf=rf, cpi=cpi),
    Portfolio('50/60_sweden', rets, {'bonds': .60, 'stocks': .50}, rf=rf, cpi=cpi),
]

In [ ]:
#| export
def _strat_key(p): return p.name.rsplit('_', 1)[0]

In [ ]:
[_strat_key(p) for p in ps]

In [ ]:
#| export
def _group_by_strategy(ps):
    strats = defaultdict(list)
    for p in ps: strats[_strat_key(p)].append(p)
    return strats

In [ ]:
_group_by_strategy(ps)

This joins all real returns by decades into common frame

In [ ]:
#| export
def _all_decade_streams(*ps):
    frames = []
    for p in ps:
        df = p.r_by_decade()
        df.columns = [f"{p.name.rsplit('_',1)[1]}_{c}" for c in df.columns]
        frames.append(df)
    return pd.concat(frames, axis=1)

In [ ]:
#| export
def strat_plot(k, v):
    df = _all_decade_streams(*v)
    med = df.median(axis=1).rename('median')
    q25 = df.quantile(.25, axis=1).rename('q25')
    q75 = df.quantile(.75, axis=1).rename('q75')
    band = pd.concat([q25, q75], axis=1)
    bg = df.hvplot(alpha=0.15, color='gray', line_width=1, legend=False, width=600, height=400, title=k, ylabel='cumulative real return', xlabel='Holding Period (years)')
    shade = band.hvplot.area(x=band.index.name or 'index', y='q25', y2='q75', alpha=0.25, color='steelblue', legend=False)
    mid = med.hvplot(line_width=3, color='crimson', label='median')
    return (bg * shade * mid).opts(shared_axes=False)

In [ ]:
#| export
def _strat_dist_plot(ps):
    strats = _group_by_strategy(ps)
    return pn.Tabs(*[(k, strat_plot(k, v)) for k,v in strats.items()])

In [ ]:
#_strat_dist_plot(ps)

### Comparing real 10 year returns forward in time across strategies

In [ ]:
#| export
def _real_return_overlay(strategy, ps, color):
    df = pd.concat({p.name: p.lost_decade() for p in ps}, axis=1)
    med = df.median(axis=1).rename(strategy)
    q25 = df.quantile(.25, axis=1).rename('q25')
    q75 = df.quantile(.75, axis=1).rename('q75')
    shade = pd.concat([q25, q75], axis=1).hvplot.area(y='q25', y2='q75', alpha=0.12, color=color, legend=False)
    return shade * med.hvplot(line_width=2.5, color=color, label=strategy)

In [ ]:
#| export
def real_return_plot(ps):
    strats = _group_by_strategy(ps)
    colors = dict(zip(strats.keys(), hv.Cycle('Category10').values))
    its = iter([_real_return_overlay(k,v,colors[k]) for k,v in strats.items()])
    overlay = next(its)
    for pl in its: overlay = overlay*pl
    return (overlay * hv.HLine(0).opts(color='black', line_width=1.5, line_dash='dotted')).opts(title='Real forward 10y returns: median ± IQR', width=700, height=450, legend_position='bottom', ylabel='Real 10y Return (ann%)')

In [ ]:
#real_return_plot(ps)

### Comparing sharpe ratios over time

In [ ]:
#| export
def rolling_sharpe(p, window=36):
    r = p.port_rets[::-1]
    s = (r.rolling(window).mean() * 12 / (r.rolling(window).std() * 12**0.5))[::-1]
    return s.dropna().rename(p.name)

In [ ]:
#| export
def _rolling_sharpe_by_strat(strategy, ps, color, window=36):
    df = pd.concat([rolling_sharpe(p, window) for p in ps], axis=1, sort=True)
    med = df.mean(axis=1).rename(strategy)
    #q25, q75 = df.quantile(.25, axis=1).rename('q25'), df.quantile(.75, axis=1).rename('q75')
    #shade = pd.concat([q25,q75], axis=1).hvplot.area(y='q25', y2='q75', alpha=0.12, color=color, legend=False)
    return med.hvplot(line_width=2.5, color=color, label=strategy)

In [ ]:
#| export
def rolling_sharpe_plot(ps):
    strats = _group_by_strategy(ps)
    colors = dict(zip(strats.keys(), hv.Cycle('Category10').values))
    its = iter([_rolling_sharpe_by_strat(k,v,colors[k], window=120) for k,v in strats.items()])
    overlay = next(its)
    for pl in its: overlay = overlay*pl
    return (overlay * hv.HLine(0).opts(color='black', line_width=1.5, line_dash='dotted')).opts(title='Rolling 10y Sharpe by strategy', width=700, height=450, legend_position='bottom', ylabel='Ann. Sharpe')

In [ ]:
#rolling_sharpe_plot(ps)

### Comparison Widget

In [ ]:
#| export
def strategy_comparison_w(ps):
    if len(ps)<=1: return None
    return pn.Column(
        pn.pane.Markdown('### Real Return Distribution by Strategy'), _strat_dist_plot(ps),
        pn.pane.Markdown('### Real Forward 10y Returns by Strategy'), real_return_plot(ps),
        pn.pane.Markdown('### Rolling Sharpe by Strategy'), rolling_sharpe_plot(ps),
    )

## Full report

In [ ]:
#| export
def _country_key(p): return p.name.rsplit('_', 1)[1] if '_' in p.name else p.name

In [ ]:
#| export
def _hier_multiselect(portfolios):
    cs = sorted(set(_country_key(p) for p in portfolios))
    ss = sorted(set(_strat_key(p) for p in portfolios))
    ns = [p.name for p in portfolios]
    cw = pn.widgets.MultiSelect(name='Countries', options=cs, value=cs, size=min(len(cs), 6))
    sw = pn.widgets.MultiSelect(name='Strategies', options=ss, value=ss, size=min(len(ss), 6))
    pw = pn.widgets.MultiSelect(name='Portfolios', options=ns, value=ns, size=min(len(ns), 8))
    def _upd_strats(e):
        new = sorted(set(_strat_key(p) for p in portfolios if _country_key(p) in set(cw.value)))
        sw.options, sw.value = new, new
    def _upd_ports(e):
        sc, ss2 = set(cw.value), set(sw.value)
        new = [p.name for p in portfolios if _country_key(p) in sc and _strat_key(p) in ss2]
        pw.options, pw.value = new, new
    cw.param.watch(_upd_strats, 'value')
    sw.param.watch(_upd_ports, 'value')
    return cw, sw, pw

In [ ]:
#| export
def _hier_select(portfolios):
    cs = sorted(set(_country_key(p) for p in portfolios))
    cw = pn.widgets.Select(name='Country', options=cs)
    init_ss = sorted(set(_strat_key(p) for p in portfolios if _country_key(p) == cs[0]))
    sw = pn.widgets.Select(name='Strategy', options=init_ss)
    init_ps = [p.name for p in portfolios if _country_key(p) == cs[0] and _strat_key(p) == init_ss[0]]
    pw = pn.widgets.Select(name='Portfolio', options=init_ps)
    def _upd_strats(e):
        new = sorted(set(_strat_key(p) for p in portfolios if _country_key(p) == cw.value))
        sw.options, sw.value = new, new[0]
    def _upd_ports(e):
        new = [p.name for p in portfolios if _country_key(p) == cw.value and _strat_key(p) == sw.value]
        pw.options, pw.value = new, new[0]
    cw.param.watch(_upd_strats, 'value')
    sw.param.watch(_upd_ports, 'value')
    return cw, sw, pw

In [ ]:
#| export
def _ports_rx(portfolios, tw, pw=None):
    def _f(clicks):
        start, end = tw[0].value
        sel = set(pw.value) if pw else {p.name for p in portfolios}
        return [p.between(start, end) for p in portfolios if p.name in sel]
    return pn.bind(_f, tw[1].param.clicks)

In [ ]:
#| export
def _single_port_rx(portfolios, tw, name_w):
    def _f(clicks):
        start, end = tw[0].value
        return next(p for p in portfolios if p.name == name_w.value).between(start, end)
    return pn.bind(_f, tw[1].param.clicks)

In [ ]:
#| export
def interactive_report(*portfolios):
    cmp_tw, ov_tw, sc_tw = time_period_w(), time_period_w(), time_period_w()
    cmp_hier = _hier_multiselect(portfolios)
    ov_hier  = _hier_select(portfolios)
    cmp_rx     = _ports_rx(portfolios, cmp_tw, cmp_hier[2])
    selected_p = _single_port_rx(portfolios, ov_tw, ov_hier[2])
    sc_rx      = _ports_rx(portfolios, sc_tw)
    window_w = pn.widgets.Select(name='Rolling Window (Years)', options=list(range(1,30)), value=1)

    summary       = pn.bind(lambda ps: pn.panel(compare(*ps)), cmp_rx)
    cum_ret_plot  = pn.bind(lambda ps: timeseries_plot(compare(*ps, metric='cum_excess_return')), cmp_rx)
    roll_ret_plot = pn.bind(lambda ps, w: timeseries_plot(compare(*ps, metric='roll_return', months=w*12), interactive_hlines=True), cmp_rx, window_w)
    real_w_plot   = pn.bind(lambda ps: timeseries_plot(compare(*ps, metric='real_w'), logy=True, is_perc=False, interactive_growth_lines=True), cmp_rx)
    drawdown_plot = pn.bind(lambda ps: timeseries_plot(compare(*ps, metric='drawdown_series')), cmp_rx)
    decade_comp   = pn.bind(lambda ps: decade_comparison_w(*ps), cmp_rx)
    lost_dec_comp = pn.bind(lambda ps: lost_decade_comparison_w(*ps), cmp_rx)

    comparison_w = pn.Column(
        pn.pane.Markdown('## Time Period & Selection'),
        pn.Row(*cmp_tw), pn.Row(*cmp_hier),
        pn.pane.Markdown('## Portfolio Stats'),
        pn.pane.Markdown('### Summary'), summary,
        pn.pane.Markdown('## Portfolio Returns'),
        pn.pane.Markdown('### Cumulative Excess Return'), cum_ret_plot,
        pn.pane.Markdown('### Rolling Excess Return'), window_w, roll_ret_plot,
        pn.pane.Markdown('### Real Wealth (Total Real Cum. Compounded Return)'), real_w_plot,
        pn.pane.Markdown('## Drawdowns'), drawdown_plot,
        pn.pane.Markdown('## Real Return comparison'), lost_dec_comp,
        pn.pane.Markdown('## Decade Comparison'), decade_comp,
    )

    overview_w = pn.Column(pn.Row(*ov_tw), pn.Row(*ov_hier), portfolio_overview(selected_p))

    strat_comp = pn.bind(lambda ps: strategy_comparison_w(ps), sc_rx)
    strat_w    = pn.Column(pn.Row(*sc_tw), strat_comp)

    return pn.Tabs(
        ('Portfolio Comparison', comparison_w),
        ('Portfolio Overview', overview_w),
        ('Strategy Comparison', strat_w),
        dynamic=True
    )

In [ ]:
#pn.panel(interactive_report(p))

In [ ]:
#| eval: false
#pn.panel(interactive_report(p)).save('report.html')

In [ ]:
# ps = [
#     Portfolio('60/40_sweden', rets, {'bonds': .40, 'stocks': .60}, rf=rf, cpi=cpi),
#     Portfolio('40/60_sweden', rets, {'bonds': .60, 'stocks': .40}, rf=rf, cpi=cpi),
# ]

In [ ]:
# server = pn.serve(lambda: interactive_report(*ps), port=6004, show=True, allow_websocket_origin=['vigilant-spiral-dances-oycg2n.solveit.pub'])

In [ ]:
# server.stop()